In [ ]:
%pip install azure-identity requests
dbutils.library.restartPython()

In [ ]:
%run ./utils_common

In [ ]:
import re
import requests
from azure.identity import ClientSecretCredential

In [ ]:
dbutils.widgets.text("catalog", "", "CATALOG")
dbutils.widgets.text("schema", "", "SCHEMA")
dbutils.widgets.text("subscription_id", "", "Azure subscription id")
dbutils.widgets.text("scope", "", "Secret scope for SP credentials")

In [ ]:
logger = setup_logger("CoveredWorkspacesDiscovery")
logging.getLogger("azure").setLevel(logging.WARNING)
logging.getLogger("azure.core").setLevel(logging.WARNING)
logging.getLogger("azure.identity").setLevel(logging.WARNING)

In [ ]:
MANAGED_RG_PREFIX = "databricks-rg-"
ARM_RG_API_VERSION = "2021-04-01"
WORKSPACE_PROVIDER_RE = re.compile(
    r"/providers/Microsoft\.Databricks/workspaces/([^/]+)$",
    re.IGNORECASE,
)


class CoveredWorkspacesClient:
    """Discover in-subscription Databricks workspaces via ARM managed RGs."""

    TABLE_NAME = "dbspend360_covered_workspaces"

    def __init__(
        self,
        subscription_id: str,
        tenant_id: str,
        client_id: str,
        client_secret: str,
        target_table: str,
        audit_table: str,
        logger=None,
    ):
        self.subscription_id = subscription_id
        self.target_table = target_table
        self.audit_table = audit_table
        self.logger = logger or logging.getLogger("CoveredWorkspacesClient")
        self.credential = ClientSecretCredential(
            tenant_id=tenant_id,
            client_id=client_id,
            client_secret=client_secret,
        )

    def _arm_token(self) -> str:
        return self.credential.get_token(
            "https://management.azure.com/.default"
        ).token

    def _list_managed_resource_groups(self):
        token = self._arm_token()
        url = (
            f"https://management.azure.com/subscriptions/{self.subscription_id}"
            f"/resourcegroups?api-version={ARM_RG_API_VERSION}"
        )
        rows = []
        while url:
            resp = requests.get(
                url,
                headers={"Authorization": f"Bearer {token}"},
                timeout=120,
            )
            resp.raise_for_status()
            payload = resp.json()
            for rg in payload.get("value", []):
                name = rg.get("name") or ""
                if name.lower().startswith(MANAGED_RG_PREFIX):
                    rows.append({
                        "rg_name": name,
                        "managed_by": rg.get("managedBy") or "",
                    })
            url = payload.get("nextLink")
        return rows

    @staticmethod
    def _workspace_name_from_managed_by(managed_by: str):
        if not managed_by:
            return None
        match = WORKSPACE_PROVIDER_RE.search(managed_by)
        return match.group(1) if match else None

    def discover(self):
        run_date = datetime.now(timezone.utc).date()
        try:
            managed_rgs = self._list_managed_resource_groups()
            self.logger.info(
                f"Found {len(managed_rgs)} managed resource groups "
                f"({MANAGED_RG_PREFIX}*) in subscription {self.subscription_id}"
            )

            rg_rows = []
            unmatched_names = []
            for rg in managed_rgs:
                ws_name = self._workspace_name_from_managed_by(rg["managed_by"])
                if ws_name:
                    rg_rows.append({
                        "workspace_name_arm": ws_name,
                        "rg_name": rg["rg_name"],
                    })
                else:
                    unmatched_names.append(rg["rg_name"])

            if unmatched_names:
                self.logger.warning(
                    f"{len(unmatched_names)} managed RGs had no parseable workspace "
                    f"name in managedBy (sample: {unmatched_names[:5]})"
                )

            if not rg_rows:
                self.logger.warning(
                    "No managed RGs with parseable workspace names; writing empty covered set."
                )
                covered_df = spark.createDataFrame(
                    [],
                    "workspace_id STRING, workspace_name STRING, subscription_id STRING, "
                    "workspace_url STRING, updated_at TIMESTAMP",
                )
            else:
                rg_df = spark.createDataFrame(rg_rows)
                workspaces_df = spark.table("system.access.workspaces_latest").select(
                    F.col("workspace_id").cast("string").alias("workspace_id"),
                    F.col("workspace_name"),
                    F.col("workspace_url"),
                    F.lower(F.col("workspace_name")).alias("workspace_name_lc"),
                )
                rg_df = rg_df.withColumn(
                    "workspace_name_lc",
                    F.lower(F.col("workspace_name_arm")),
                )
                joined = (
                    rg_df.alias("rg")
                    .join(
                        workspaces_df.alias("wl"),
                        on="workspace_name_lc",
                        how="inner",
                    )
                    .select(
                        F.col("wl.workspace_id"),
                        F.col("wl.workspace_name"),
                        F.lit(self.subscription_id).alias("subscription_id"),
                        F.col("wl.workspace_url"),
                    )
                    .dropDuplicates(["workspace_id"])
                )
                covered_df = joined.withColumn(
                    "updated_at", F.current_timestamp()
                )

                matched_lc = set(
                    row.workspace_name_lc
                    for row in rg_df.select("workspace_name_lc").distinct().collect()
                )
                bridged_lc = set(
                    row.workspace_name_lc
                    for row in joined.select(
                        F.lower(F.col("workspace_name")).alias("workspace_name_lc")
                    ).distinct().collect()
                )
                name_miss = len(matched_lc - bridged_lc)
                if name_miss:
                    self.logger.warning(
                        f"{name_miss} ARM workspace names did not match "
                        f"system.access.workspaces_latest (case-insensitive)"
                    )

            row_count = covered_df.count()
            covered_df.write.format("delta").mode("overwrite").saveAsTable(
                self.target_table
            )
            self.logger.info(
                f"Wrote {row_count} covered workspaces to {self.target_table}"
            )

            self._emit_usage_degradation_signal()

            log_audit_run(
                self.audit_table,
                self.TABLE_NAME,
                run_date,
                run_date,
                "SUCCESS",
                row_count,
                f"subscription={self.subscription_id}",
            )
        except Exception as e:
            msg = str(e)[:1000]
            self.logger.error(f"Covered-workspace discovery failed: {msg}")
            try:
                log_audit_run(
                    self.audit_table,
                    self.TABLE_NAME,
                    run_date,
                    run_date,
                    "FAILED",
                    0,
                    msg,
                )
            except Exception:
                self.logger.error("Failed to write FAILED audit entry")
            raise

    def _emit_usage_degradation_signal(self):
        """Warn when workspaces with recent DBU usage are absent from covered set."""
        try:
            missing_df = spark.sql(f"""
                WITH recent_usage AS (
                    SELECT DISTINCT CAST(workspace_id AS STRING) AS workspace_id
                    FROM system.billing.usage
                    WHERE usage_date >= date_sub(current_date(), 30)
                ),
                covered AS (
                    SELECT DISTINCT workspace_id FROM {self.target_table}
                )
                SELECT u.workspace_id
                FROM recent_usage u
                LEFT JOIN covered c ON u.workspace_id = c.workspace_id
                WHERE c.workspace_id IS NULL
            """)
            missing_count = missing_df.count()
            if missing_count > 0:
                sample = [
                    row.workspace_id
                    for row in missing_df.limit(10).collect()
                ]
                self.logger.warning(
                    f"DEGRADATION: {missing_count} workspace(s) with DBU usage in the "
                    f"last 30 days are absent from {self.target_table}. "
                    f"Sample workspace_ids: {sample}"
                )
            else:
                self.logger.info(
                    "All workspaces with recent DBU usage are present in covered set."
                )
        except Exception as e:
            self.logger.warning(f"Usage degradation check failed (non-fatal): {e}")

In [ ]:
class CoveredWorkspacesApp:

    def __init__(self):
        catalog = dbutils.widgets.get("catalog")
        schema = dbutils.widgets.get("schema")
        subscription_id = dbutils.widgets.get("subscription_id").strip()
        scope = dbutils.widgets.get("scope").strip()

        if not subscription_id:
            raise ValueError("subscription_id widget must be set for ARM discovery.")
        if not scope:
            raise ValueError("scope widget must be set for SP credentials.")

        tenant_id = dbutils.secrets.get(scope, "tenant_id")
        client_id = dbutils.secrets.get(scope, "client_id")
        client_secret = dbutils.secrets.get(scope, "client_secret")

        self.client = CoveredWorkspacesClient(
            subscription_id=subscription_id,
            tenant_id=tenant_id,
            client_id=client_id,
            client_secret=client_secret,
            target_table=build_table_fqn(catalog, schema, "dbspend360_covered_workspaces"),
            audit_table=build_table_fqn(catalog, schema, "dbspend360_audit_log"),
            logger=logger,
        )

    def run(self):
        self.client.discover()

In [ ]:
app = CoveredWorkspacesApp()
app.run()